## W8-3 Stockout and reorder (Ibrahim Khalil Umar)

*Section for `08_deep_dive_analysis.ipynb`. Self-contained: it finds the merged data itself and asserts its checks pass. Full analysis, report and outputs: `standalone_question_reports/w8_3_stockout_reorder/`.*

**Question.** Which of the 180 stock pairs are likely to run out, and what reorder point should each use? Parameters are fitted on 2019 to 2022 and judged on 2023 to 2025.

> Sales scale assumed (Q10 unresolved); results rest on this assumption. No holding or ordering cost data, so no policy is called cost-optimal.

In [1]:
import os, pathlib
import numpy as np, pandas as pd
from scipy.stats import norm, rankdata
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
w83_SEED, w83_REPS, w83_WARM = 42, 500, 60
w83_TRAIN_END = pd.Timestamp("2022-12-31"); w83_HERE = pathlib.Path.cwd()
def w83_find():
    env = os.environ.get("CADETX_MERGED")
    if env and pathlib.Path(env).exists(): return pathlib.Path(env)
    for p in [w83_HERE, *w83_HERE.parents]:
        c = p / "week-02" / "reports" / "week_02" / "merged"
        if c.exists(): return c
    raise SystemExit("Merged datasets not found: set CADETX_MERGED or run inside the repository checkout.")
w83_M = w83_find(); w83_LABEL = "Sales scale assumed (Q10 unresolved); results rest on this assumption"
w83_checks = []
def w83_check(name, ok, detail=""):
    w83_checks.append(dict(Check_Item=name, Actual=str(detail), Status="PASS" if ok else "FAIL")); assert ok, f"{name}: {detail}"
K = ["product_id", "branch_id"]
w83_L = pd.read_csv(w83_M / "stock_ledger_enriched.csv", low_memory=False, parse_dates=["movement_date"])
w83_I = pd.read_csv(w83_M / "inventory_with_ledger.csv")
w83_P = pd.read_csv(w83_M / "purchase_orders_full.csv", low_memory=False, parse_dates=["order_date"], usecols=["product_id", "branch_id", "supplier_id", "po_status", "order_date", "lead_time_actual_days"])
w83_O = w83_L[w83_L.movement_type == "OUT"]; w83_dates = pd.date_range(w83_O.movement_date.min(), w83_O.movement_date.max(), freq="D")
w83_idx = pd.MultiIndex.from_frame(w83_I[K]); NP = 180
D = w83_O.groupby(K + ["movement_date"]).quantity.sum().unstack("movement_date").reindex(columns=w83_dates, fill_value=0).fillna(0).reindex(w83_idx, fill_value=0).values.astype(float)
T = D.shape[1]; t_cut = int((w83_dates <= w83_TRAIN_END).sum())
w83_check("demand matrix total equals ledger OUT total", int(D.sum()) == int(w83_O.quantity.sum()), int(D.sum()))
w83_check("test window is 744 days", T - t_cut == 744, T - t_cut)
Rr = w83_P[w83_P.po_status.eq("Received") & w83_P.lead_time_actual_days.notna()]
def w83_emp(Rx):
    g = Rx.groupby(K).lead_time_actual_days; e = [g.get_group(tuple(k)).values.astype(int) for k in w83_idx]
    c = np.array([len(x) for x in e]); M = np.zeros((NP, c.max()), int)
    for i, x in enumerate(e): M[i, :len(x)] = x
    return e, c, M
emp_tr, cnt_tr, E_tr = w83_emp(Rr[Rr.order_date <= w83_TRAIN_END]); emp_te, cnt_te, E_te = w83_emp(Rr[Rr.order_date > w83_TRAIN_END])
w83_check("every pair has at least 400 training and 200 test lead-time lines", cnt_tr.min() >= 400 and cnt_te.min() >= 200, f"{cnt_tr.min()} / {cnt_te.min()}")
ltm = np.array([x.mean() for x in emp_tr]); lts = np.array([x.std(ddof=1) for x in emp_tr])
mu = D[:, :t_cut].mean(1); sd = D[:, :t_cut].std(1, ddof=1)
within = (Rr.groupby("supplier_id").lead_time_actual_days.var() * Rr.groupby("supplier_id").size()).sum() / len(Rr)
w83_between = 1 - within / Rr.lead_time_actual_days.var()
print(f"share of lead-time variance between suppliers: {w83_between:.3f}")
w83_check("more than 90% of lead-time variance is between suppliers", w83_between > .90, round(w83_between, 3))

share of lead-time variance between suppliers: 0.934


In [2]:
def w83_sim(s, S, reps, seed, t_start, t_end, t_count, Em, cn_pair):
    N = NP * reps; rng = np.random.default_rng(seed); s_ = np.tile(s, reps); S_ = np.tile(S, reps); pr = np.tile(np.arange(NP), reps)
    B = 64; pipe = np.zeros((N, B)); onhand = S_.copy(); onorder = np.zeros(N); unmet = np.zeros(N); area = np.zeros(N); rows = np.arange(N); cn = cn_pair[pr]
    for t in range(t_start, t_end):
        sl = t % B; a = pipe[:, sl].copy(); pipe[:, sl] = 0; onhand += a; onorder -= a
        d = np.tile(D[:, t], reps); sold = np.minimum(d, onhand); onhand -= sold
        if t >= t_count: unmet += d - sold; area += onhand
        pos = onhand + onorder; q = S_ - pos; m = (pos <= s_) & (q > 0); u = rng.random(N)
        if m.any():
            r = rows[m]; lead = Em[pr[r], np.minimum((u[r] * cn[r]).astype(int), cn[r] - 1)]; pipe[r, (t + lead) % B] += q[r]; onorder[r] += q[r]
    return unmet.reshape(reps, NP), (area / (t_end - t_count)).reshape(reps, NP)
sA = w83_I.reorder_level.values.astype(float); SA = w83_I.max_stock.values.astype(float); gap = SA - sA
s95 = np.ceil(mu * ltm + 1.645 * np.sqrt(ltm * sd**2 + mu**2 * lts**2))
test_dem = D[:, t_cut:].sum(1); cost = w83_I.unit_cost.values.astype(float); t_start = t_cut - w83_WARM
uA, aA = w83_sim(sA, SA, w83_REPS, w83_SEED, t_start, T, t_cut, E_te, cnt_te)
uB, aB = w83_sim(s95, s95 + gap, w83_REPS, w83_SEED, t_start, T, t_cut, E_te, cnt_te)
frA = 1 - uA.sum(1) / test_dem.sum(); frB = 1 - uB.sum(1) / test_dem.sum(); pfA = (1 - uA / test_dem).mean(0); pfB = (1 - uB / test_dem).mean(0)
W83 = pd.DataFrame({"policy": ["A existing", "B-LT 95% (fitted 2019-2022)"], "fill_rate": [frA.mean(), frB.mean()],
                    "fill_p2_5": [np.percentile(frA, 2.5), np.percentile(frB, 2.5)], "fill_p97_5": [np.percentile(frA, 97.5), np.percentile(frB, 97.5)],
                    "pairs_below_95": [int((pfA < .95).sum()), int((pfB < .95).sum())],
                    "avg_stock_value_M": [(aA * cost).sum(1).mean() / 1e6, (aB * cost).sum(1).mean() / 1e6], "label": w83_LABEL})
print(W83.round(4).to_string(index=False))
w83_check("fitted B-LT 95% beats A in all 500 out-of-sample paths", bool((frB > frA).all()), f"{(frB > frA).mean():.3f}")
w83_check("A leaves between 70 and 85 pairs below 95% fill in the test period (full notebook: 78)", 70 <= W83.pairs_below_95.iloc[0] <= 85, int(W83.pairs_below_95.iloc[0]))
w83_check("B-LT 95% leaves no pair below 95% fill", W83.pairs_below_95.iloc[1] == 0, int(W83.pairs_below_95.iloc[1]))
# risk score from training data only
zA = (sA - mu * ltm) / np.sqrt(ltm * sd**2 + mu**2 * lts**2); p_cycle = 1 - norm.cdf(zA); y = pfA < .95
r = rankdata(p_cycle); n1 = y.sum(); auc = (r[y].sum() - n1 * (n1 + 1) / 2) / (n1 * (len(y) - n1)); r2 = rankdata(-sA); auc_b = (r2[y].sum() - n1 * (n1 + 1) / 2) / (n1 * (len(y) - n1))
print(f"stockout-risk score AUC {auc:.3f} (baseline: reorder_level alone {auc_b:.3f})")
w83_check("training-data risk score separates the pairs that ran short (AUC above 0.9)", auc > .9, round(auc, 3))
pairs = pd.DataFrame({"product_id": w83_I.product_id, "branch_id": w83_I.branch_id, "existing_reorder_level": sA, "reorder_point_95": s95, "order_up_to_95": s95 + gap,
                      "backtest_fill_existing": pfA.round(4), "backtest_fill_recommended": pfB.round(4), "predicted_cycle_stockout_prob": p_cycle.round(4), "label": w83_LABEL})
w83_check("one reorder point per pair, order-up-to above it", len(pairs) == 180 and (pairs.order_up_to_95 > pairs.reorder_point_95).all())
print(pairs.head(6).to_string(index=False))

                     policy  fill_rate  fill_p2_5  fill_p97_5  pairs_below_95  avg_stock_value_M                                                                 label
                 A existing     0.9384     0.9348      0.9419              78           397.1404 Sales scale assumed (Q10 unresolved); results rest on this assumption
B-LT 95% (fitted 2019-2022)     0.9929     0.9914      0.9941               0           498.7680 Sales scale assumed (Q10 unresolved); results rest on this assumption
stockout-risk score AUC 0.966 (baseline: reorder_level alone 0.965)
product_id branch_id  existing_reorder_level  reorder_point_95  order_up_to_95  backtest_fill_existing  backtest_fill_recommended  predicted_cycle_stockout_prob                                                                 label
      P001    DEL001                    59.0             101.0           382.0                  0.9764                     0.9934                         0.3574 Sales scale assumed (Q10 unresolved); r

In [3]:
fig, ax = plt.subplots(figsize=(7.5, 4.4)); ax.scatter(p_cycle * 100, pfA * 100, s=16, c=np.where(y, "#C8791A", "#1B6B66"), alpha=.8); ax.axhline(95, color="#8A8F98", ls="--", lw=.9)
ax.set_xlabel("Predicted cycle stockout probability from 2019-2022 data, %"); ax.set_ylabel("Back-test fill under existing levels, 2023-2025, %"); ax.set_title(f"Training-data risk score vs what happened (AUC {auc:.2f})")
for s_ in ["top", "right"]: ax.spines[s_].set_visible(False)
fig.text(.5, -.02, w83_LABEL, ha="center", fontsize=8.5, color="#4A5163"); plt.tight_layout(); plt.savefig("w8_3_section_risk.png", dpi=150, bbox_inches="tight"); plt.show()
w83_check("chart written", pathlib.Path("w8_3_section_risk.png").exists())

**Reading it.** Existing levels would have left 78 of 180 pairs below 95% fill in 2023 to 2025 (about 93.8% of demand met). A reorder point fitted on 2019 to 2022 with the lead-time-aware formula at 95% service meets about 99.3% with none below 95%, for about 25.6% more stock value, and beats the existing levels in every simulated path. The risk score ranks the weak pairs well (AUC 0.97) but no better than the existing reorder level alone. Most lead-time spread is which supplier fills the order (93% of variance).

**Data dictionary update (`week-08/data_dictionary_week08.csv`).** Supplier fields: `supplier_lead_days` is the promised lead; `delivery_delay_days` runs -3 to +7 and about 15% late for every supplier; `reliability_score` does not track lateness; `china_tax_id` is Unusable.

In [4]:
w83_dd = w83_find().parents[3] / "week-08" / "data_dictionary_week08.csv"
if w83_dd.exists():
    dd = pd.read_csv(w83_dd)
    w83_check("data dictionary marks china_tax_id Unusable", (dd[dd.column == "china_tax_id"].usability == "Unusable").all())
    w83_check("data dictionary marks reliability_score Usable with caution", (dd[dd.column == "reliability_score"].usability == "Usable with caution").all())
pd.DataFrame(w83_checks).to_csv("validation_report_week08_w8_3_section.csv", index=False)
print(len(w83_checks), "checks,", sum(c["Status"] == "PASS" for c in w83_checks), "pass")

10 checks, 10 pass
